# Jokes clustering and classification using SBERT

Antoine de Chabannes Curton la Palice, Hugo Viana

## Abstract

l'abstract ici

## 1. Introduction

L'intro ici

## 2. Materials and methods

Dependencies

In [ ]:
import os
import numpy as np
import pandas as pd
import kagglehub
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# Classifiers
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import SGDClassifier

# NN classifier
import torch
import torch.nn as nn
import lightning as L
import torchmetrics
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

### 2.1. Dataset

Le dataset ici

In [ ]:
# Download latest version
path = kagglehub.dataset_download("abhinavmoudgil95/short-jokes")

df = pd.read_csv(os.path.join(path, "shortjokes.csv"))
print(df.head())

Using Colab cache for faster access to the 'short-jokes' dataset.
   ID                                               Joke
0   1  [me narrating a documentary about narrators] "...
1   2  Telling my daughter garlic is good for you. Go...
2   3  I've been going through a really rough period ...
3   4  If I could have dinner with anyone, dead or al...
4   5     Two guys walk into a bar. The third guy ducks.


### 2.2. Jokes embedding

Embedding avec SBERT

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

df["embedding"] = df["Joke"].apply(lambda x: np.array(model.encode(x)))

print(df.head())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   ID                                               Joke  \
0   1  [me narrating a documentary about narrators] "...   
1   2  Telling my daughter garlic is good for you. Go...   
2   3  I've been going through a really rough period ...   
3   4  If I could have dinner with anyone, dead or al...   
4   5     Two guys walk into a bar. The third guy ducks.   

                                           embedding  
0  [-0.0019190103, -0.04273604, -0.017304454, -0....  
1  [-0.07587159, 0.06581424, 0.006111781, 0.06642...  
2  [-0.094529964, -0.031642836, 0.091890045, 0.05...  
3  [-0.077876724, 0.023547389, -0.10157586, 0.007...  
4  [0.04891309, -0.086067915, -0.041825227, -0.04...  


### 2.3. Clustering

1. Choix du nombre de clusters
2. Choix de l'algo (K-Means)
3. Evaluation et Interprétati

### 2.4. Classification




1.   Labelisation des données avec les clusters extraits
2.   modèle de classification
3.   optimisation des paramètres
4.   entrainement du modèle



#### Multinomial Naïve Bayesian

#### Stochastic Gradient Descent classifier

#### Feed-Forward Neural Network

In [ ]:
class JokeDataset(Dataset):
    def __init__(self, df, embedding_col="embedding", label_col="label"):
        self.df = df.copy()
        self.embedding_col = embedding_col
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        joke = torch.tensor(
            self.df.iloc[idx][self.embedding_col],
            dtype=torch.float32,
            requires_grad=False
        )
        label = torch.tensor(self.df.iloc[idx][self.label_col], dtype=torch.long)
        return joke, label

class JokeDataModule(LightningDataModule):
    def __init__(self, df, embedding_col="embedding", label_col="label"):
        super().__init__()
        self.df = df.copy()
        self.embedding_col = embedding_col
        self.label_col = label_col
        self.scaler = None

    def prepare_data(self):
        embeddings = self.df[self.embedding_col].values.reshape(-1, 1)
        self.scaler = StandardScaler()
        scaled_embeddings = self.scaler.fit_transform(embeddings)

        self.df[self.embedding_col] = scaled_embeddings.tolist()

    def setup(self, stage=None):
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            self.df[self.embedding_col],
            self.df[self.label_col],
            test_size=0.2,
            random_state=42)

        X_train, X_val, y_train, y_val = train_test_split(
            X_train_val,
            y_train_val,
            test_size=0.125,
            random_state=42)

        self.train_df = pd.DataFrame({
            'embedding': X_train.tolist(),
            'label': y_train.tolist()})

        self.val_df = pd.DataFrame({
            'embedding': X_val.tolist(),
            'label': y_val.tolist()})

        self.test_df = pd.DataFrame({
            'embedding': X_test.tolist(),
            'label': y_test.tolist()})

    def train_dataloader(self):
        return DataLoader(
            JokeDataset(self.train_df, self.embedding_col, self.label_col),
            batch_size=32,
            shuffle=True)

    def val_dataloader(self):
        return DataLoader(
            JokeDataset(self.val_df, self.embedding_col, self.label_col),
            batch_size=32,
            shuffle=False)

    def test_dataloader(self):
        return DataLoader(
            JokeDataset(self.test_df, self.embedding_col, self.label_col),
            batch_size=32,
            shuffle=False)

In [ ]:
class NNClassifier(L.LightningModule):
    def __init__(self, input_dim, num_classes, n_layers, n_units, learning_rate):
        super().__init__()

        self.save_hyperparameters()

        self.lr = learning_rate

        self.activation_fn = nn.ReLu()

        self.layers = nn.Sequential()
        self.layers.add_module('input_layer', nn.Linear(input_dim, n_units))
        self.layers.add_module('input_activation', self.activation_fn)
        for i in range(n_layers - 1):
            self.layers.add_module(f'hidden_layer_{i+1}', nn.Linear(n_units, n_units))
            self.layers.add_module(f'hidden_activation_{i+1}', self.activation_fn)
        self.out = nn.Linear(n_units, num_classes)

        self.criterion = nn.CrossEntropyLoss()

        self.optimizer = torch.optim.Adam

        self.train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.train_precision = torchmetrics.Precision(task="multiclass", num_classes=num_classes, average='macro')
        self.train_recall = torchmetrics.Recall(task="multiclass", num_classes=num_classes, average='macro')
        self.train_f1 = torchmetrics.F1Score(task="multiclass", num_classes=num_classes, average='macro')

        self.val_acc = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.val_precision = torchmetrics.Precision(task="multiclass", num_classes=num_classes, average='macro')
        self.val_recall = torchmetrics.Recall(task="multiclass", num_classes=num_classes, average='macro')
        self.val_f1 = torchmetrics.F1Score(task="multiclass", num_classes=num_classes, average='macro')

        self.test_acc = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.test_precision = torchmetrics.Precision(task="multiclass", num_classes=num_classes, average='macro')
        self.test_recall = torchmetrics.Recall(task="multiclass", num_classes=num_classes, average='macro')
        self.test_f1 = torchmetrics.F1Score(task="multiclass", num_classes=num_classes, average='macro')



    def forward(self, x):
        x = self.layers(x)
        x = self.out(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        y = y.view(-1)
        y_hat = self.forward(x)
        loss = self.criterion(y_hat, y)
        if len(y) > 1:
            acc = self.train_acc(y_hat.argmax(dim=1), y)
            precision = self.train_precision(y_hat.argmax(dim=1), y)
            recall = self.train_recall(y_hat.argmax(dim=1), y)
            f1 = self.train_f1(y_hat.argmax(dim=1), y)

            self.log("train_loss", loss, prog_bar=True)
            self.log("train_acc", acc, on_epoch=True, prog_bar=True, sync_dist=True)
            self.log("train_precision", precision, on_epoch=True, prog_bar=True, sync_dist=True)
            self.log("train_recall", recall, on_epoch=True, prog_bar=True, sync_dist=True)
            self.log("train_f1", f1, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y = y.view(-1)
        y_hat = self.forward(x)
        loss = self.criterion(y_hat, y)
        if len(y) > 1:
            acc = self.val_acc(y_hat.argmax(dim=1), y)
            precision = self.val_precision(y_hat.argmax(dim=1), y)
            recall = self.val_recall(y_hat.argmax(dim=1), y)
            f1 = self.val_f1(y_hat.argmax(dim=1), y)

            self.log("val_loss", loss, prog_bar=True)
            self.log("val_acc", acc, on_epoch=True, prog_bar=True, sync_dist=True)
            self.log("val_precision", precision, on_epoch=True, prog_bar=True, sync_dist=True)
            self.log("val_recall", recall, on_epoch=True, prog_bar=True, sync_dist=True)
            self.log("val_f1", f1, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        y = y.view(-1)
        y_hat = self.forward(x)
        loss = self.criterion(y_hat, y)
        if len(y) > 1:
            acc = self.test_acc(y_hat.argmax(dim=1), y)
            precision = self.test_precision(y_hat.argmax(dim=1), y)
            recall = self.test_recall(y_hat.argmax(dim=1), y)
            f1 = self.test_f1(y_hat.argmax(dim=1), y)

            self.log("test_loss", loss, prog_bar=True)
            self.log("test_acc", acc, on_epoch=True, prog_bar=True, sync_dist=True)
            self.log("test_precision", precision, on_epoch=True, prog_bar=True, sync_dist=True)
            self.log("test_recall", recall, on_epoch=True, prog_bar=True, sync_dist=True)
            self.log("test_f1", f1, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def configure_optimizers(self):
        optimizer =  self.optimizer(self.parameters(), lr=self.lr)
        return optimizer

In [ ]:
def objective(trial):
        n_layer = trial.suggest_int('n_layer', 1, 5)
        n_units = trial.suggest_categorical('n_units', [16, 32, 64, 128, 256, 512])
        learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)

        model = NNClassifier(input_dim=?, num_classes=?, n_layers=n_layers, n_units=n_units, learning_rate=learning_rate)
        dm = JokeDataModule(df=?)

        logger = TensorBoardLogger(f"tb_log", name=f"trial_{trial.number}")

        pruning_callback = PyTorchLightningPruningCallback(
            trial, monitor="val_loss"
        )

        trainer = L.Trainer(
            max_epochs=1000,
            callbacks=[EarlyStopping(monitor='val_loss', patience=5), pruning_callback],
            logger=logger,
            enable_checkpointing=False
        )

        trainer.fit(model, dm)
        val_result = trainer.validate(model, datamodule=dm)
        val_acc = val_result[0]['val_acc']
        trainer.test(model, datamodule=dm)

        return val_acc

In [ ]:
study = optuna.create_study(
    direction='maximize',
    storage='sqlite:///study.db',
    study_name='NNClassifier',
    load_if_exists=True)

study.optimize(objective, n_trials=100)

## 3. Results

Résultats du clustering, quels types de blagues ressortent ?
Performances de la classification

### 3.1 Clustering

Résultats du clustering, quels types de blagues ressortent ?

### 3.2. Classification

Performance du modèle de classification

## 4. Discussion

La conclusion plus ou moins

## 5. Future works

Perspectives futures

## References

1.   Abhinav Moudgil. **Short Jokes**. https://www.kaggle.com/datasets/abhinavmoudgil95/short-jokes
2.   Nils Reimers, Iryna Gurevych. **Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks**. Proceedings of the 2019 Conference on Empirical Methods in Natural Language Processing, 2019. https://arxiv.org/abs/1908.10084
3.   Alina Petukhovaa, João P. Matos-Carvalhoa, Nuno Fachada. **Text clustering with LLM embeddings**. International Journal of Cognitive Computing in Engineering, Volume 6, 2025. https://doi.org/10.1016/j.ijcce.2024.11.004

